# Figure-1-style reproducibility analysis

This notebook investigates whether different diffusion model widths generate statistically similar CAMELS LH sample sets at the same training-data size. The intended comparison is:

- U64 vs U128
- U64 vs U256
- U128 vs U256

The notebook is manifest-driven. It does not assume private Great Lakes paths are committed to the repo. Put the real generated `.npy` paths in a local manifest, then run the cells below.

## Required inputs

The manifest is a JSON list. Each row should describe one generated sample set:

```json
{
  "run_name": "lh_u64_d2p10",
  "arch": "u64",
  "dataset_tag": "2^10",
  "dataset_size": 1024,
  "sample_path": "../results/tables/samples/lh_u64_d2p10_seed123.npy"
}
```

`dataset_size` should mean the number of 2D training slices if that is what you want on the x-axis. If your manifest instead uses `actual_2d` or `target_2d`, the notebook can use that too. For CAMELS LH with `zthin=4`, each 3D simulation contributes 32 2D slices, so `n_samples=32` simulations gives `32 * 32 = 1024` 2D training slices.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from simdiff_eval.io import as_nchw, load_npy
from simdiff_eval.metrics import reproducibility_summary

# For public repo testing, this template may contain placeholder paths.
# For your Great Lakes runs, point this to an ignored local manifest instead.
MANIFEST_PATH = PROJECT_DIR / "configs/templates/reproducibility_manifest_template.json"
SAMPLE_ROOT = PROJECT_DIR / "results/tables/samples"
SEED = 123
NBINS_PK = 25
X_FIELD = "auto"  # auto prefers dataset_size, then actual_2d, then target_2d

PAIRS = [("u64", "u128"), ("u64", "u256"), ("u128", "u256")]

## Score definition

For each dataset size and architecture pair, compare generated sample set `a` and generated sample set `b`.

Power-spectrum mismatch:

$$
E_{P(k)} = \frac{1}{N_k}\sum_k \left|\log_{10}\left(\frac{\bar P_a(k)}{\bar P_b(k)}\right)\right|
$$

One-point mean mismatch:

$$
E_\mu = |\mu_a - \mu_b|
$$

One-point standard-deviation mismatch:

$$
E_\sigma = |\sigma_a - \sigma_b|
$$

Combined error:

$$
E = E_{P(k)} + E_\mu + E_\sigma
$$

Reproducibility score:

$$
S = \frac{1}{1 + E}
$$

So `S=1` means perfect agreement under these diagnostics. Lower values mean the generated distributions differ more. This is a transparent project diagnostic; it is Figure-1-style, not necessarily the exact score used in a paper.

In [ ]:
manifest = json.loads(MANIFEST_PATH.read_text())
manifest_df = pd.DataFrame(manifest)
manifest_df

In [ ]:
def resolve_sample_path(row, seed=SEED):
    if row.get("sample_path"):
        raw = str(row["sample_path"]).format(seed=seed, run_name=row["run_name"])
        path = Path(raw)
        return path if path.is_absolute() else MANIFEST_PATH.parent / path
    return SAMPLE_ROOT / f"{row['run_name']}_seed{seed}.npy"


def row_dataset_size(row, x_field=X_FIELD):
    if x_field != "auto":
        return float(row[x_field])
    for key in ("dataset_size", "actual_2d", "target_2d"):
        if key in row:
            return float(row[key])
    raise KeyError(f"{row['run_name']} needs dataset_size, actual_2d, or target_2d")


manifest_df["plot_dataset_size"] = [row_dataset_size(row) for row in manifest]
manifest_df["resolved_sample_path"] = [str(resolve_sample_path(row)) for row in manifest]
manifest_df["sample_exists"] = [Path(p).exists() for p in manifest_df["resolved_sample_path"]]
manifest_df[["run_name", "arch", "dataset_tag", "plot_dataset_size", "sample_exists", "resolved_sample_path"]]

## Compute pairwise reproducibility

This cell skips missing files so you can run it before all jobs finish. Once U64, U128, and U256 samples exist for the same `dataset_tag`, it will compute the three pairwise scores.

In [ ]:
def score_from_metrics(metrics):
    error = (
        metrics["pk_log10_mae_between_sets"]
        + metrics["mean_abs_mean_diff"]
        + metrics["std_abs_diff"]
    )
    return error, 1.0 / (1.0 + error)


by_size = {}
for row in manifest:
    by_size.setdefault(str(row["dataset_tag"]), {})[str(row["arch"])] = row

rows = []
for dataset_tag, arch_rows in sorted(by_size.items(), key=lambda item: row_dataset_size(next(iter(item[1].values())))):
    for arch_a, arch_b in PAIRS:
        if arch_a not in arch_rows or arch_b not in arch_rows:
            continue
        path_a = resolve_sample_path(arch_rows[arch_a])
        path_b = resolve_sample_path(arch_rows[arch_b])
        if not path_a.exists() or not path_b.exists():
            print(f"skip missing {dataset_tag} {arch_a}:{arch_b}")
            continue

        samples = {
            arch_a: as_nchw(load_npy(path_a)),
            arch_b: as_nchw(load_npy(path_b)),
        }
        metrics = reproducibility_summary(samples, nbins=NBINS_PK)[0]
        error, score = score_from_metrics(metrics)
        rows.append({
            "dataset_tag": dataset_tag,
            "dataset_size": row_dataset_size(arch_rows[arch_a]),
            "pair": f"{arch_a}_vs_{arch_b}",
            "run_a": arch_rows[arch_a]["run_name"],
            "run_b": arch_rows[arch_b]["run_name"],
            "reproducibility_error": error,
            "reproducibility_score": score,
            **metrics,
        })

scores_df = pd.DataFrame(rows)
scores_df

In [ ]:
if scores_df.empty:
    print("No complete pairwise comparisons yet. Check the sample_exists column above.")
else:
    fig, ax = plt.subplots(figsize=(7, 5))
    colors = {"u64_vs_u128": "red", "u64_vs_u256": "blue", "u128_vs_u256": "limegreen"}
    markers = {"u64_vs_u128": "o", "u64_vs_u256": "s", "u128_vs_u256": "^"}
    labels = {
        "u64_vs_u128": "UNet-64 vs UNet-128",
        "u64_vs_u256": "UNet-64 vs UNet-256",
        "u128_vs_u256": "UNet-128 vs UNet-256",
    }

    for pair, sub in scores_df.groupby("pair"):
        sub = sub.sort_values("dataset_size")
        ax.plot(
            sub["dataset_size"],
            sub["reproducibility_score"],
            marker=markers.get(pair, "o"),
            color=colors.get(pair),
            lw=2.5,
            label=labels.get(pair, pair),
        )

    ax.set_xscale("log", base=2)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("dataset size")
    ax.set_ylabel("reproducibility score")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    plt.show()